In [ ]:
!pip install torch transformers bitsandbytes accelerate
!pip install langchain faiss-cpu sentence-transformers
!pip install camelot-py[pdf] pymupdf gradio pandas
!pip install gradio==3.50.2 websockets==10.4
!pip uninstall -y langchain
!pip install langchain==0.1.16
!pip install langchain-community==0.0.36
!pip install langchain-text-splitters==0.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 5.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 14.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
Found existing installation: langchain 1.2.0
Uninstalling langchain-1.2.0:
  Successfully uninstalled langchain-1.2.0
  Using cached langchain-0.1.16-py3-none-any.whl.metadata (13 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.0.38
    Uninstalling langchain-community-0.0.38:
      Successfully uninstalled langchain-community-0.0.38
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.0.2
    Uninstalling langchain-text-splitters-0.0.2:
      Successfully uninstalled langchain-text-splitters-0.0.2


In [ ]:
PDF_PATH = "Your PDF Path"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

def generate_answer(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=False
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
import camelot

def extract_tables(pdf_path):
    tables = camelot.read_pdf(pdf_path, pages="all", flavor="lattice")

    table_store = []
    for i, t in enumerate(tables):
        df = t.df
        table_store.append({
            "table_id": f"table_{i}",
            "page": t.page,
            "headers": list(df.iloc[0]),
            "rows": df.iloc[1:].values.tolist(),
            "dataframe": df,
            "summary": df.head(2).to_string()
        })
    return table_store


In [ ]:


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


In [ ]:
import fitz
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.embeddings import HuggingFaceEmbeddings
# from langchain.vectorstores import FAISS

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def build_vector_db(text, table_store):
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_text(text)

    docs = []
    metas = []

    for c in chunks:
        docs.append(c)
        metas.append({"type": "text"})

    for t in table_store:
        docs.append(t["summary"])
        metas.append({
            "type": "table",
            "table_id": t["table_id"],
            "page": t["page"]
        })

    embedder = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    db = FAISS.from_texts(docs, embedder, metadatas=metas)
    return db


In [ ]:

# from model_loader import generate_answer
# from table_utils import extract_tables
# from rag_utils import extract_text, build_vector_db



print("🔹 Extracting tables...")
table_store = extract_tables(PDF_PATH)

print("🔹 Extracting text...")
text = extract_text(PDF_PATH)

print("🔹 Building vector DB...")
db = build_vector_db(text, table_store)

def get_table_by_id(table_id):
    return next(t for t in table_store if t["table_id"] == table_id)

def chatbot(query):
    docs = db.similarity_search(query, k=5)

    context = ""
    selected_table = None

    for d in docs:
        if d.metadata["type"] == "table" and selected_table is None:
            selected_table = get_table_by_id(d.metadata["table_id"])
        else:
            context += d.page_content + "\n"

    prompt = f"""
You are an assistant for technical PDFs.
Answer ONLY from the context.

Context:
{context}

Question:
{query}

Answer:
"""

    answer = generate_answer(prompt)

    if selected_table:
        return (
            answer,
            selected_table["dataframe"],
            {
                "table_id": selected_table["table_id"],
                "page": selected_table["page"],
                "headers": selected_table["headers"],
                "rows": selected_table["rows"]
            }
        )

    return answer, None, None




🔹 Extracting tables...


/usr/local/lib/python3.12/dist-packages/camelot/utils.py:1217: UserWarning:   (536.9205, 539.4224999999999) does not lie in column range (68.85591765372985, 536.5796370967741)
  warnings.warn(


🔹 Extracting text...
🔹 Building vector DB...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import gradio as gr
gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(label="Ask a question"),
    outputs=[
        gr.Textbox(label="Text Answer"),
        gr.Dataframe(label="Table from PDF"),
        gr.JSON(label="Stored Table (Internal)")
    ],
    title=""
).launch(debug = True)

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://9161a75100413ade9d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
